In [1]:
import torch
import numpy as np
import pandas as pd
from torch.optim import AdamW
from sklearn.metrics import log_loss
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from optional_fine_tune import BertForCompareSentences, DfToDataset, train_epoch, eval_model

D:\Codding\Education\NLP\Quora Question Pairs\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv("data/train.csv")
train = train.dropna(subset=['question1', 'question2'])
train = train.reset_index(drop=True)

In [3]:
targets = train['is_duplicate']

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

Используем устройство: cuda


In [ ]:
k_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_loss_scores = []

for fold, (train_idx, val_idx) in enumerate(k_fold.split(train, targets)):
    X_train = train.iloc[train_idx]
    X_val = train.iloc[val_idx]

    y_train = X_train['is_duplicate']
    y_val = X_val['is_duplicate']

    train_dataset = DfToDataset(X_train, tokenizer)
    val_dataset = DfToDataset(X_val, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    model = BertForCompareSentences(MODEL_NAME).to(device)

    EPOCHS = 3
    loss_fn = CrossEntropyLoss().to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    metric = log_loss

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
    )

    best_fold_loss = 0
    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, optimizer, train_loader, loss_fn, scheduler, device)
        val_loss = eval_model(model, metric, val_loader, device)

        print(f"Эпоха {epoch + 1}/{EPOCHS} | LogLoss: {val_loss:.4f}")

        if val_loss < best_fold_loss:
            best_fold_loss = val_loss

            model_path = f"models/best_bert_fold_{fold + 1}.pt"
            torch.save(model.state_dict(), model_path)
            print(f"Веса модели сохранены в {model_path}")

    print(f"Лучший LogLoss для фолда {fold + 1}: {best_fold_loss:.4f}")
    cv_loss_scores.append(best_fold_loss)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11110.44it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print(f"ИТОГОВЫЙ СРЕДНИЙ FINE-TUNED F1-SCORE: {np.mean(cv_loss_scores):.4f}")